# 12 — Multimodal Prompt Engineering

## Scenario
Northstar automatically extracts data from invoice images submitted by users. 
However, users often provide contradictory text (e.g., typing "Here is my invoice for $800", but attaching an image that says "$500").

**The Problem:** We cannot trust the user's text, nor can we assume the OCR/Model will always perfectly read the image. If there is a contradiction, the model should *not* guess or compromise; it must flag the uncertainty so we can escalate to a human.


In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab12 import CASES, InvoiceExtraction, build_requests, claimed_amount, route


def show_request(request):
    print("SYSTEM:\n", request.system)
    for message in request.messages:
        if message.text:
            print(f"{message.role.upper()}:\n{message.text}")
        for part in message.parts:
            print(f"{message.role.upper()} PART:", part)

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Step 1: The Multimodal Contradiction

We will pass the image (showing $500) AND a user prompt (claiming $800) to the model simultaneously. 
We use Pydantic to force the model to explicitly evaluate the evidence rather than just returning a string.


In [ ]:
request = next(r for r in build_requests() if r.case_id == "i12/claim-800")
show_request(request)
print("IMAGE PART:", request.messages[0].parts[0])
response = client.generate(request)
print("RECORDED RESPONSE:", response.text)
extraction = InvoiceExtraction.model_validate_json(response.text)
print("PARSED:", extraction)
assert route(extraction, claimed_amount(CASES[0]["message"])) == "human_review"


## Step 2: Reconcile the user claim with extracted evidence

The application also escalates a mismatch even if a replayed model response incorrectly says there is no contradiction.


In [ ]:
for case in CASES[1:]:
    request = next(r for r in build_requests() if r.case_id == f"i12/{case['id']}")
    show_request(request)
    response = client.generate(request)
    extraction = InvoiceExtraction.model_validate_json(response.text)
    decision = route(extraction, claimed_amount(case["message"]))
    print("RECORDED RESPONSE:", response.text)
    print("PARSED:", extraction, "DECISION:", decision)
    assert decision == case["expected"]


## Conclusion

By combining native multimodality (passing pixels directly to the model) with Structured Outputs (Pydantic schemas), we can build highly robust data pipelines that fail safely. Instead of guessing when faced with contradictory evidence, the model structures its uncertainty, allowing the application layer to trigger a human-in-the-loop escalation.


## Takeaway
The recorded invoice run routed the $800 contradiction and missing-claim case to human review, while the matching $500 claim was auto-processed.


## References
- [Core Concepts & Workflow](README.md#core-concepts--workflow)
- [Deep dive](README.md#deep-dive)
- [Lab walkthrough](README.md#lab-walkthrough)
